In [1]:
import multiprocessing
import numpy as np
import pandas as pd
import tensorflow_hub as hub
import tensorflow as tf
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from gensim.models import Doc2Vec
from gensim.models.doc2vec import TaggedDocument

In [2]:
# Embedding ~6M reviews in this dataset using SBERT, USE and Doc2Vec requires ~90-120min for each of them
df = pd.read_csv('data/dataset.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6417106 entries, 0 to 6417105
Data columns (total 5 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   app_id        int64 
 1   app_name      object
 2   review_text   object
 3   review_score  int64 
 4   review_votes  int64 
dtypes: int64(3), object(2)
memory usage: 244.8+ MB


In [4]:
df_pos = df.loc[df['review_score'] == 1][:30000]
df_neg = df.loc[df['review_score'] == -1][:30000]
df = pd.concat([df_pos, df_neg], ignore_index=True)

In [5]:
def clean(texts):
    return texts.strip()

In [6]:
''' 
SBERT can capture semantic relationships and support multilingual data,
it has high quality embedding but the embedding process may took time. 
The feature space of SBERT is 384 and has dense results.
'''

' \nSBERT can capture semantic relationships and support multilingual data,\nit has high quality embedding but the embedding process may took time. \nThe feature space of SBERT is 384 and has dense results.\n'

In [7]:
texts = df['review_text'].astype(str).apply(clean).tolist()
sbert = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

batch_size, l = 512, len(texts)
embedded_sbert = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using SBERT'):
    batch = texts[i:min(l, i+batch_size)]
    emb_sbert = sbert.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    embedded_sbert.append(emb_sbert)

Embedding using SBERT: 100%|█████████████████████████████████████████████████████████| 118/118 [01:29<00:00,  1.32it/s]


In [8]:
embedded_sbert = pd.DataFrame(np.vstack(embedded_sbert), columns=[f'emb{i}' for i in range(384)])
df_sbert = pd.concat([df.iloc[:, :3], embedded_sbert, df.iloc[:, 3:]], axis=1)

In [9]:
df_sbert.to_csv('results/df_sbert_balanced.csv', index=False)

In [10]:
'''
Universal Sentence Encoder can capture semantic relationships and also support multilingual datasets like SBERT.
It has lower quality embeddings but faster speed than SBERT.
The feature space of it is 512.
'''

'\nUniversal Sentence Encoder can capture semantic relationships and also support multilingual datasets like SBERT.\nIt has lower quality embeddings but faster speed than SBERT.\nThe feature space of it is 512.\n'

In [11]:
texts = df['review_text'].astype(str).apply(clean).tolist()
use = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')

batch_size, l = 512, len(texts)
embedded_use = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using USE'):
    batch = texts[i:min(l, i+batch_size)]
    emb_use = use(batch)
    embedded_use.append(emb_use)


Embedding using USE: 100%|███████████████████████████████████████████████████████████| 118/118 [01:24<00:00,  1.40it/s]


In [12]:
embedded_use = pd.DataFrame(np.vstack(embedded_use), columns=[f'emb{i}' for i in range(512)])
df_use = pd.concat([df.iloc[:, :3], embedded_use, df.iloc[:, 3:]], axis=1)

In [13]:
df_use.to_csv('results/df_use_balanced.csv', index=False)

In [14]:
'''
Doc2Vec need training on the Steam datasets (~6M reviews), which is good.
It captures some semantics relationship and weak in multilingual environments.
'''

'\nDoc2Vec need training on the Steam datasets (~6M reviews), which is good.\nIt captures some semantics relationship and weak in multilingual environments.\n'

In [5]:
# There are ~6M Steam reviews, so it is better to apply simple cleaning in this case. The model can be more robust
def clean_doc2vec(texts):
    return texts.lower().strip()

def read_data(texts):
    for i, text in enumerate(texts):
        yield TaggedDocument(words=text.split(), tags=[str(i)])

In [6]:
texts = df['review_text'].astype(str).apply(clean_doc2vec)

doc2vec = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=10,
    workers=multiprocessing.cpu_count(),
    dm=0,
    epochs=1,
    seed=42
)

doc2vec.build_vocab(read_data(texts))

In [7]:
EPOCHS, l = 30, len(texts)
alpha_start, alpha_end = 0.025, 0.0001
alpha_delta = (alpha_start - alpha_end) / EPOCHS
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    documents = read_data(texts)
    documents = tqdm(documents, total=l, desc="Docs")

    alpha = alpha_start - epoch * alpha_delta
    doc2vec.alpha = alpha
    doc2vec.min_alpha = alpha 
    
    doc2vec.train(
        documents,
        total_examples=doc2vec.corpus_count,
        epochs=1,
        start_alpha=alpha,
        end_alpha=alpha
    )


Epoch 1/30


Docs: 100%|████████████████████████████████████████████████████████████████████| 60000/60000 [00:06<00:00, 9102.13it/s]



Epoch 2/30


Docs: 100%|████████████████████████████████████████████████████████████████████| 60000/60000 [00:06<00:00, 9575.41it/s]



Epoch 3/30


...

Docs: 100%|████████████████████████████████████████████████████████████████████| 60000/60000 [00:07<00:00, 8049.55it/s]



Epoch 29/30


Docs: 100%|████████████████████████████████████████████████████████████████████| 60000/60000 [00:07<00:00, 8192.97it/s]



Epoch 30/30


Docs: 100%|████████████████████████████████████████████████████████████████████| 60000/60000 [00:07<00:00, 8425.77it/s]


In [8]:
doc2vec.save('results/doc2vec_balanced.model')

In [9]:
doc_vectors = np.array([doc2vec.dv[str(i)] for i in range(l)])
embedded_doc2vec = pd.DataFrame(doc_vectors, columns=[f'emb{i}' for i in range(doc2vec.vector_size)])
df_doc2vec = pd.concat([df.iloc[:, :3], embedded_doc2vec, df.iloc[:, 3:]], axis=1)

In [10]:
df_doc2vec.to_csv('results/df_doc2vec_balanced.csv', index=False)